In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import plotly.express as px
import plotly.graph_objects as go

weights = "data/masks/weights_canada.nc"
temp = "data/canada/data_0.nc"
weights = xr.open_dataset(weights)
temp = xr.open_dataset(temp)

<frozen importlib._bootstrap>:241: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject


In [4]:
import sys, numpy, netCDF4, xarray
print(sys.executable)
print(sys.version)
print("numpy", numpy.__version__)
print("netCDF4", netCDF4.__version__)
print("xarray", xarray.__version__)


/opt/homebrew/Caskroom/mambaforge/base/envs/titanic-ml/bin/python
3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:52:34) [Clang 18.1.8 ]
numpy 2.3.3
netCDF4 1.7.3
xarray 2025.12.0


In [15]:
def visualize_climate_nc(weights, region_name=None):
    da = temp["t2m"]
    da = da - 273.15
    if "year" in da.dims:
        da = da.isel(year=-1)
    elif "valid_time" in da.dims:
        da = da.isel(valid_time=-1)

    da = da.transpose("latitude", "longitude")
    da = da.sortby("latitude")
    da = da.sortby("longitude")

    wt = weights["final_weights"] if "final_weights" in weights else weights
    wt = wt.sortby("latitude")
    wt = wt.sortby("longitude")
    wt = wt.interp(latitude=da.latitude, longitude=da.longitude, method="nearest").fillna(0.0)
    mask = wt > 0

    da_region = da.where(mask).dropna("latitude", how="all").dropna("longitude", how="all")
    if da_region.size == 0:
        raise ValueError("Le masque ne recouvre aucune cellule de la grille temperature.")

    min_temp = float(da.min().item())
    max_temp = float(da.max().item())
    lat_min = float(da_region.latitude.min())
    lat_max = float(da_region.latitude.max())
    lon_min = float(da_region.longitude.min())
    lon_max = float(da_region.longitude.max())

    pad = 0.5
    lat_desc = bool(da_region.latitude.values[0] > da_region.latitude.values[-1])
    lon_desc = bool(da_region.longitude.values[0] > da_region.longitude.values[-1])
    lat_range = [lat_max + pad, lat_min - pad] if lat_desc else [lat_min - pad, lat_max + pad]
    lon_range = [lon_max + pad, lon_min - pad] if lon_desc else [lon_min - pad, lon_max + pad]

    title = f"Carte actuelle des temperatures ({region_name})" if region_name else "Carte actuelle des temperatures"
    fig = px.imshow(
        da_region,
        x=da_region["longitude"],
        y=da_region["latitude"],
        color_continuous_scale="RdBu_r",
        origin="lower",
        title=title,
        labels={"color": "Temp (°C)"},
        range_color=[min_temp, max_temp]
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, title=None)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, title=None)
    fig.update_xaxes(range=lon_range, autorange=False)
    fig.update_yaxes(range=lat_range, autorange=False)
    return fig

fig1 = visualize_climate_nc(weights, region_name="Canada")
fig1